In [ ]:
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "retail_operations_dev")
catalog = dbutils.widgets.get("catalog").strip()

order_items = spark.table(
    f"{catalog}.silver.order_items_current"
)

products = spark.table(
    f"{catalog}.silver.products_current"
)

# A product ID must match at most one catalogue record.
duplicate_products = (
    products
    .groupBy("product_id")
    .count()
    .filter(F.col("count") > 1)
)

assert duplicate_products.limit(1).count() == 0, (
    "The product catalogue contains duplicate product IDs."
)

In [ ]:
enriched_items = (
    order_items.alias("orders")
    .join(
        products.alias("products"),
        F.col("orders.product_id") == F.col("products.product_id"),
        how = 'left',
    )
    .select(
        "orders.*",
        F.col("products.product_name").alias("current_product_name"),
        F.col("products.category").alias("current_category"),
        F.col("products.catalog_price").alias("current_catalog_price"),
        F.col("products.currency").alias("catalog_currency"),
        F.col("products.is_active").alias("currently_active"),
        F.col("products.updated_at").alias("catalog_updated_at"),
        F.col("products.product_id").isNull().alias(
            "product_missing"
        ),
    )
)

display(
    enriched_items
    .filter(F.col("order_id") == "O10001")
    .select(
        "order_id",
        "product_id",
        "current_product_name",
        "current_category",
        "quantity",
        "unit_price",
        "current_catalog_price",
        "line_amount",
        "product_missing",
    )
)

In [ ]:
# Select the original order columns from the enriched result.
original_columns = order_items.columns

enriched_original_fields = enriched_items.select(
    *original_columns
)

# Compare in both directions, including duplicate occurrences.
assert order_items.exceptAll(enriched_original_fields).limit(1).count() == 0, (
    "Some original order-item records were lost or changed."
)

assert enriched_original_fields.exceptAll(order_items).limit(1).count() == 0, (
    "The enrichment added or changed original order-item records."
)

# Inspect catalogue coverage.
display(
    enriched_items
    .groupBy("product_missing")
    .count()
)

print("Enrichment preserved the original order-item records.")

In [ ]:
enriched_items_table = (
    f"{catalog}.gold.order_items_enriched"
)

(
    enriched_items
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(enriched_items_table)
)
print(f"Saved: {enriched_items_table}")